# CAPs Clustering Analysis on tDCS data

K-means clustering with different K values on the concatenated z-scored time-series. <br>
This notebook requires running `01_time_series_extraction.ipynb` first to generate the concatenated time-series data. <br>
This notebook does not include the procedure for choosing the best K.

### Imports and Configuration

In [7]:
import os
from dotenv import load_dotenv
from nilearn.maskers import NiftiLabelsMasker
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from munkres import Munkres
import time

In [8]:
# Load private paths from .env file
load_dotenv()

subs = ['Y01', 'Y02', 'Y03', 'Y04', 'Y05', 'Y06', 'Y07', 'Y08', 'Y09','Y10',
        'Y11', 'Y12', 'Y13', 'Y14', 'Y15', 'Y16', 'Y17','Y18', 'Y19', 'Y20']
sessions = [1, 2, 3]
runs = [2]
sessions_dict = {1: "Unilateral", 2: "Bilateral", 3: "Sham"}
runs_dict = {1: "tDCS Off", 2: "tDCS On"}

base_dir = os.environ['BASE_DIR']
downsampled_ts_dir = os.environ['DOWNSAMPLED_TS_DIR']
input_dir = os.environ['INPUT_DIR']

runs_str = "run-ON"
output_dir = os.path.join(base_dir, f"coactivation_patterns_{runs_str}")
os.makedirs(output_dir, exist_ok=True)

### Load atlas masker and pre-computed time-series data

In [9]:
atlas_dir = os.environ['ATLAS_DIR']
atlas_filename = os.path.join(atlas_dir, "atlas_combined_tpl-MNI152NLin2009cAsym_res-02_T1w.nii.gz")
atlas_infos = pd.read_csv(os.path.join(atlas_dir, "atlas_combined_labels.csv"))

atlas_masker = NiftiLabelsMasker(labels_img=atlas_filename, mask_img=None, standardize=False, standardize_confounds=True, memory="nilearn_cache", verbose=0)
atlas_masker.fit()

,labels_img,'C:/Users/user/Desktop/...cAsym_res-02_T1w.nii.gz'
,labels,None
,lut,None
,background_label,0
,mask_img,None
,smoothing_fwhm,None
,standardize,False
,standardize_confounds,True
,high_variance_confounds,False
,detrend,False
,low_pass,None


In [10]:
# Load the concatenated z-scored time-series and labels from time_series_analysis output
CAP_TS = np.load(os.path.join(output_dir, f"CAP_TS_{runs_str}_zscored.npy"))
DATA_LABELS = np.load(os.path.join(output_dir, f"DATA_LABELS_{runs_str}.npy"))

print(f"CAP_TS shape: {CAP_TS.shape}")
print(f"DATA_LABELS shape: {DATA_LABELS.shape}")

CAP_TS shape: (22740, 1060)
DATA_LABELS shape: (22740,)


### Clustering
k=2-10

In [11]:
from sklearn.metrics.pairwise import cosine_similarity

k_values = range(2,11)

sil_scores = []
inertias = []
k_loop_start =time.time()
for k_idx, k in enumerate(k_values):
    print(f"Starting k-means for k={k}")
    k_start = time.time()
    CAPdata = dict()
    CAPdata['subject_list'] = subs
    CAPdata['sessions'] = sessions
    CAPdata['runs'] = runs
    CAPdata['masker'] = atlas_masker
    CAPdata['DATA_LABELS'] = DATA_LABELS

    clust_out_dir = os.path.join(output_dir, f"caps_nclust-{k}")
    os.makedirs(clust_out_dir, exist_ok=True)

    kmeans = KMeans(n_clusters=k, random_state=42, n_init="auto").fit(CAP_TS)

    score = silhouette_score(CAP_TS, kmeans.labels_, sample_size=int(CAP_TS.shape[0]*0.25))
    inertias.append(kmeans.inertia_)
    sil_scores.append(score)

    np.save(os.path.join(clust_out_dir, f"CAP_clusters_without_pairing_nclust-{k}.npy"), {
        'labels': kmeans.labels_,
        'centers': kmeans.cluster_centers_,
        'data_labels': DATA_LABELS
    })

    cluster_centers = atlas_masker.inverse_transform(kmeans.cluster_centers_)
    cluster_centers.to_filename(os.path.join(clust_out_dir, f"CAPcenter_without_pairing_nclust-{k}.nii.gz"))

    
    # re order CAPS
    np.random.seed(42)
    r = cosine_similarity(kmeans.cluster_centers_)
    m = Munkres()
    indexes = m.compute(r)
    print(indexes)
    cap_order_paired = []


    for ii in np.arange(k):
        print(cap_order_paired)
        print(indexes[ii])
        if (indexes[ii][0] not in cap_order_paired):
            cap_order_paired.append(indexes[ii][0])
        if (indexes[ii][1] not in cap_order_paired):
            cap_order_paired.append(indexes[ii][1])

    print(cap_order_paired)
    labels_raw = kmeans.labels_
    centers_raw = kmeans.cluster_centers_

    CAPdata['labels_raw'] = labels_raw
    CAPdata['centers_raw'] = centers_raw
    CAPdata['cap_order_paired'] = cap_order_paired

    labels_paired = np.zeros(labels_raw.shape)
    centers_paired = np.zeros(centers_raw.shape)
    centers_average = np.zeros(centers_raw.shape)

    for new_idx, ll in enumerate(cap_order_paired):
        labels_paired[labels_raw==ll] = new_idx
        centers_paired[new_idx, :] = centers_raw[ll, :]
        centers_average[new_idx,:] = np.nanmean(CAP_TS[labels_raw==ll, :], axis=0)

    CAPdata['labels_paired'] = labels_paired
    CAPdata['centers_paired'] = centers_paired
    CAPdata['centers_average'] = centers_average


    cluster_centers = atlas_masker.inverse_transform(centers_paired)
    cluster_centers.to_filename(os.path.join(clust_out_dir, f"CAPcenter_paired_nclust-{k}.nii.gz"))

    cluster_centers = atlas_masker.inverse_transform(centers_average)
    cluster_centers.to_filename(os.path.join(clust_out_dir, f"CAPcenter_average_nclust-{k}.nii.gz"))

    np.save(os.path.join(clust_out_dir, f"CAP_DATA_nclust-{k}.npy"), CAPdata)

print(f"Finished!!, it took {time.time()-k_loop_start:.2f} seconds")

Starting k-means for k=2
[(0, 1), (1, 0)]
[]
(0, 1)
[0, 1]
(1, 0)
[0, 1]
Starting k-means for k=3
[(0, 2), (1, 0), (2, 1)]
[]
(0, 2)
[0, 2]
(1, 0)
[0, 2, 1]
(2, 1)
[0, 2, 1]
Starting k-means for k=4
[(0, 1), (1, 0), (2, 3), (3, 2)]
[]
(0, 1)
[0, 1]
(1, 0)
[0, 1]
(2, 3)
[0, 1, 2, 3]
(3, 2)
[0, 1, 2, 3]
Starting k-means for k=5
[(0, 1), (1, 4), (2, 0), (3, 2), (4, 3)]
[]
(0, 1)
[0, 1]
(1, 4)
[0, 1, 4]
(2, 0)
[0, 1, 4, 2]
(3, 2)
[0, 1, 4, 2, 3]
(4, 3)
[0, 1, 4, 2, 3]
Starting k-means for k=6
[(0, 5), (1, 2), (2, 1), (3, 4), (4, 3), (5, 0)]
[]
(0, 5)
[0, 5]
(1, 2)
[0, 5, 1, 2]
(2, 1)
[0, 5, 1, 2]
(3, 4)
[0, 5, 1, 2, 3, 4]
(4, 3)
[0, 5, 1, 2, 3, 4]
(5, 0)
[0, 5, 1, 2, 3, 4]
Starting k-means for k=7
[(0, 6), (1, 5), (2, 1), (3, 4), (4, 3), (5, 2), (6, 0)]
[]
(0, 6)
[0, 6]
(1, 5)
[0, 6, 1, 5]
(2, 1)
[0, 6, 1, 5, 2]
(3, 4)
[0, 6, 1, 5, 2, 3, 4]
(4, 3)
[0, 6, 1, 5, 2, 3, 4]
(5, 2)
[0, 6, 1, 5, 2, 3, 4]
(6, 0)
[0, 6, 1, 5, 2, 3, 4]
Starting k-means for k=8
[(0, 7), (1, 4), (2, 3), (3, 2), (4, 1)